# Hands-On LangChain #9: Output Parsing

Companion to [the full article](https://todatabeyond.com/blog/hands-on-langchain-for-llm-applications-development-output-parsing).

Default execution tests **real LangChain parsers and Pydantic validation** on explicitly scripted response-text fixtures, without API keys or model downloads. These are parsing/integration tests, NOT live LLM extraction or a reproduction of hosted outputs. The complete ten-step original hosted workflow is maintained in an opt-in appendix. Original blog output panels are historical; runtime-verified results are labelled separately.

## 1. Install dependencies

In [ ]:
!pip install -q "langchain-core>=1.0,<2" "langchain-classic>=1.0,<2" "langchain-openai>=1.0,<2" "pydantic>=2,<3" python-dotenv

## 2. Original review and actual prompt formatting
The original example dictionary uses a string for price_value; the extraction workflow below uses an array of strings. JSON booleans are lower-case true/false; Python dictionary booleans are True/False.

In [ ]:
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

In [ ]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a JSON array of strings.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_template(review_template)
messages = prompt_template.format_messages(text=customer_review)
assert prompt_template.input_variables == ["text"]
assert customer_review in messages[0].content
print("Prompt variables:", prompt_template.input_variables)
print("Original review preserved in formatted message: True")

## 3. JSON-looking text is still a string
This explicitly scripted fixture follows the archived response, with valid straight JSON quotes. It is not model inference. The expected string/dictionary error is caught so execution continues.

In [ ]:
import json
from langchain_core.messages import AIMessage
fixture = {"gift": True, "delivery_days": 2, "price_value": [
    "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
]}
response = AIMessage(content=json.dumps(fixture))
assert isinstance(response.content, str)
print("Fixture response content type:", type(response.content).__name__)

In [ ]:
# The response content is a string, not a dictionary.
# Keep this intentional error handled so the notebook can continue.
try:
    response.content.get("gift")
except AttributeError as error:
    print(f"AttributeError: {error}")

## 4. Original StructuredOutputParser, current import

ResponseSchema.type guides prompt formatting; it does NOT validate field types. StructuredOutputParser checks expected keys and parses JSON, including Markdown fences. Its JSON utility may repair incomplete JSON. The comment-bearing format-instruction example is schema guidance, not literal valid JSON. We set boolean/integer/array labels rather than the source's default string labels.

In [ ]:
from langchain_classic.output_parsers import ResponseSchema
from langchain_classic.output_parsers import StructuredOutputParser


gift_schema = ResponseSchema(name="gift", type="boolean",
                             description="Was the item purchased\
                             as a gift for someone else? \
                             Answer True if yes,\
                             False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days", type="integer",
                                      description="How many days\
                                      did it take for the product\
                                      to arrive? If this \
                                      information is not found,\
                                      output -1.")
price_value_schema = ResponseSchema(name="price_value", type="array",
                                    description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    JSON array of strings.")

response_schemas = [gift_schema, 
                    delivery_days_schema,
                    price_value_schema]

In [ ]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()
assert all(key in format_instructions for key in fixture)
output_dict = output_parser.parse(response.content)
assert output_dict == fixture
assert output_parser.parse("```json\n" + response.content + "\n```") == fixture
print("Parsed fixture:", output_dict)
print("Result:", type(output_dict).__name__, "gift:", output_dict["gift"])
print("Markdown-fenced JSON: parsed")

legacy_string_days = dict(fixture, delivery_days="2")
legacy_result = output_parser.parse(json.dumps(legacy_string_days))
assert isinstance(legacy_result["delivery_days"], str)
print("Legacy parser keeps string delivery_days:", repr(legacy_result["delivery_days"]))

In [ ]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a JSON array of strings.

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review, 
                                format_instructions=format_instructions)

In [ ]:
from langchain_core.exceptions import OutputParserException
assert customer_review in messages[0].content and format_instructions in messages[0].content
try:
    output_parser.parse('{"gift": true}')
except OutputParserException:
    print("Missing legacy schema key: OutputParserException verified")
else:
    raise AssertionError("Missing schema keys must fail")

## 5. Modern strict typed validation

Use a strict Pydantic model to enforce a boolean gift, integer delivery_days >= -1, and list of strings price_value. Extra fields are forbidden. PydanticOutputParser itself can tolerate incomplete JSON; this application first uses json.loads to require a complete valid JSON value, then sends the normalized value to the real typed parser. Markdown-fenced strings are intentionally rejected by this strict entry point. Shape validation does not prove extraction accuracy or grounding in the review.

In [ ]:
from langchain_core.exceptions import OutputParserException
from pydantic import BaseModel, ConfigDict, Field, StrictBool, StrictInt, StrictStr
from langchain_core.output_parsers import PydanticOutputParser

class ReviewExtraction(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")
    gift: StrictBool = Field(description="Whether purchased as a gift; false if unknown")
    delivery_days: StrictInt = Field(ge=-1, description="Delivery days; -1 if unknown")
    price_value: list[StrictStr] = Field(description="Verbatim sentences about price or value")

typed_parser = PydanticOutputParser(pydantic_object=ReviewExtraction)
def parse_review_strict(text: str) -> ReviewExtraction:
    # Reject incomplete JSON and fences before LangChain's tolerant JSON utility.
    raw = json.loads(text)
    if not isinstance(raw, dict):
        raise ValueError("Expected one JSON object")
    return typed_parser.parse(json.dumps(raw, allow_nan=False))

typed_review = parse_review_strict(response.content)
assert typed_review.model_dump() == fixture
assert type(typed_review.gift) is bool and type(typed_review.delivery_days) is int
assert type(typed_review.price_value) is list
print("Strict typed result:", typed_review.model_dump())
print("Field types: bool, int, list[str]")

In [ ]:
invalid_cases = {
    "missing key": json.dumps({"gift": True}),
    "string delivery days": json.dumps(dict(fixture, delivery_days="2")),
    "string boolean": json.dumps(dict(fixture, gift="true")),
    "string price list": json.dumps(dict(fixture, price_value="expensive")),
    "nonstring price entry": json.dumps(dict(fixture, price_value=[1])),
    "invalid day sentinel": json.dumps(dict(fixture, delivery_days=-2)),
    "extra field": json.dumps(dict(fixture, unexpected="value")),
    "malformed JSON": "not JSON",
    "truncated JSON": response.content[:-1],
    "array instead of object": "[]",
    "markdown fence": "```json\n" + response.content + "\n```",
}
for name, text in invalid_cases.items():
    try:
        parse_review_strict(text)
    except (ValueError, OutputParserException):
        print("Rejected:", name)
    else:
        raise AssertionError(f"Invalid case accepted: {name}")
unknown = parse_review_strict('{"gift": false, "delivery_days": -1, "price_value": []}')
assert unknown.model_dump() == {"gift": False, "delivery_days": -1, "price_value": []}
print("Unknown values: False, -1, [] accepted")

## 6. Real prompt → scripted response → parser composition

The RunnableLambda response is explicitly scripted. This verifies that the review and format instructions reach the response stage and that the strict parser follows it, not that a model extracted correct information.

In [ ]:
from langchain_core.runnables import RunnableLambda
seen = []
def scripted_response(prompt_value):
    seen.append(prompt_value.to_messages()[0].content)
    return response.content
typed_prompt = ChatPromptTemplate.from_template(
    "Extract gift, delivery_days, and price_value from this review.\n{format_instructions}\nReview: {text}"
).partial(format_instructions=typed_parser.get_format_instructions())
chain = typed_prompt | RunnableLambda(scripted_response) | RunnableLambda(parse_review_strict)
chain_result = chain.invoke({"text": customer_review})
assert chain_result.model_dump() == fixture
assert len(seen) == 1 and customer_review in seen[0]
assert "delivery_days" in seen[0] and "schema" in seen[0]
print("Prompt → scripted response → strict parser: verified (no inference)")
print("All output-parsing assertions passed.")

## 7. Complete optional hosted original workflow

All ten original code steps follow with current imports/invoke, explicit model and credentials setup, JSON array wording, type hints, and a caught intentional AttributeError. Set OPENAI_API_KEY and OPENAI_CHAT_MODEL locally and enable the flag to make paid hosted calls. Only set OPENAI_TEMPERATURE when the model supports it. These hosted calls were not executed; historical blog responses are not asserted as current outputs. Never commit credentials or use private reviews without permission. A prompt is not a security boundary and parsing does not verify factual extraction.

In [ ]:
RUN_HOSTED_EXAMPLE = False

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print({
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
})

In [ ]:
if RUN_HOSTED_EXAMPLE:
    customer_review = """\
    This leaf blower is pretty amazing.  It has four settings:\
    candle blower, gentle breeze, windy city, and tornado. \
    It arrived in two days, just in time for my wife's \
    anniversary present. \
    I think my wife liked it so much she was speechless. \
    So far I've been the only one using it, and I've been \
    using it every other morning to clear the leaves on our lawn. \
    It's slightly more expensive than the other leaf blowers \
    out there, but I think it's worth it for the extra features.
    """
    
    review_template = """\
    For the following text, extract the following information:
    
    gift: Was the item purchased as a gift for someone else? \
    Answer True if yes, False if not or unknown.
    
    delivery_days: How many days did it take for the product \
    to arrive? If this information is not found, output -1.
    
    price_value: Extract any sentences about the value or price,\
    and output them as a JSON array of strings.
    
    Format the output as JSON with the following keys:
    gift
    delivery_days
    price_value
    
    text: {text}
    """

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_core.prompts import ChatPromptTemplate
    
    prompt_template = ChatPromptTemplate.from_template(review_template)
    print(prompt_template)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    import os
    from langchain_openai import ChatOpenAI
    from dotenv import load_dotenv
    load_dotenv()
    for name in ("OPENAI_API_KEY", "OPENAI_CHAT_MODEL"):
        if not os.environ.get(name):
            raise RuntimeError(f"Set {name} locally before running hosted examples.")
    llm_model = os.environ["OPENAI_CHAT_MODEL"]
    model_options = {}
    if os.environ.get("OPENAI_TEMPERATURE") is not None:
        model_options["temperature"] = float(os.environ["OPENAI_TEMPERATURE"])
    messages = prompt_template.format_messages(text=customer_review)
    chat = ChatOpenAI(model=llm_model, **model_options)
    response = chat.invoke(messages)
    print(response.content)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    print(type(response.content))

In [ ]:
if RUN_HOSTED_EXAMPLE:
    # The response content is a string, not a dictionary.
    # Keep this intentional error handled so the notebook can continue.
    try:
        response.content.get("gift")
    except AttributeError as error:
        print(f"AttributeError: {error}")

In [ ]:
if RUN_HOSTED_EXAMPLE:
    from langchain_classic.output_parsers import ResponseSchema
    from langchain_classic.output_parsers import StructuredOutputParser
    
    
    gift_schema = ResponseSchema(name="gift", type="boolean",
                                 description="Was the item purchased\
                                 as a gift for someone else? \
                                 Answer True if yes,\
                                 False if not or unknown.")
    delivery_days_schema = ResponseSchema(name="delivery_days", type="integer",
                                          description="How many days\
                                          did it take for the product\
                                          to arrive? If this \
                                          information is not found,\
                                          output -1.")
    price_value_schema = ResponseSchema(name="price_value", type="array",
                                        description="Extract any\
                                        sentences about the value or \
                                        price, and output them as a \
                                        JSON array of strings.")
    
    response_schemas = [gift_schema, 
                        delivery_days_schema,
                        price_value_schema]

In [ ]:
if RUN_HOSTED_EXAMPLE:
    output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
    format_instructions = output_parser.get_format_instructions()
    print(format_instructions)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    review_template_2 = """\
    For the following text, extract the following information:
    
    gift: Was the item purchased as a gift for someone else? \
    Answer True if yes, False if not or unknown.
    
    delivery_days: How many days did it take for the product\
    to arrive? If this information is not found, output -1.
    
    price_value: Extract any sentences about the value or price,\
    and output them as a JSON array of strings.
    
    text: {text}
    
    {format_instructions}
    """
    
    prompt = ChatPromptTemplate.from_template(template=review_template_2)
    
    messages = prompt.format_messages(text=customer_review, 
                                    format_instructions=format_instructions)

In [ ]:
if RUN_HOSTED_EXAMPLE:
    response = chat.invoke(messages)
    output_dict = output_parser.parse(response.content)
    print(output_dict)